<img src="https://s3-us-west-2.amazonaws.com/public.notion-static.com/7fa59d58-ba42-4de1-840a-e2e31ab9ce3b/ba416d9e-ee4d-4b7a-a9fe-a506333af2b7.png" alt="Abstract Banner" width="100%" height="200" style="object-fit: cover; border-radius: 8px;">

by: Elmar Leonard, Muhammad Rafi Andrianto and Valencia

# **Section 0: Project Initialization**

## **0.1 Importing Library**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import TransformerMixin

from sklearn.model_selection import cross_validate, GridSearchCV, learning_curve

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.model_selection import TimeSeriesSplit

from sklearn.preprocessing import FunctionTransformer, RobustScaler, OneHotEncoder
from category_encoders import BinaryEncoder

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import LinearSVC, SVC

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, BaggingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve

from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import average_precision_score

import pickle
import shap
import joblib
import copy
from datetime import date
import os

pd.set_option('display.max_columns', None)

c:\Users\elmar\anaconda3\envs\Python_3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **0.2 Global Configuration**

In [2]:
DATA_DIR = 'Dataset' 
timestamp_columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'shipping_limit_date']

RANDOM_STATE = 42

df = pd.read_csv(f'../{DATA_DIR}/Cleaned Data/cleaned.csv', parse_dates=timestamp_columns)
delivered = df[df["order_status"]=="delivered"]

pd.set_option('display.max_columns', None)
print(f"Dataframe size: {df.shape}")
print(f"Delivered Dataframe size: {delivered.shape}")

Dataframe size: (111777, 39)
Delivered Dataframe size: (108566, 39)


# **Section 1: Dataset Preparation**

## **1.1 splitting the data**

Prior to splitting the dataset, it is necessary to establish the target variable. Because the objective is to predict delivery punctuality, a boolean column was created to clearly flag whether or not each order is late.

In [3]:
delivered['is_late'] = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']

The dataset was partitioned into training, validation, and testing sets using a 60/20/20 ratio. To preserve the chronological integrity of the historical data, a time-based split was utilized instead of randomized sample selection.

In [4]:
order_purchase = delivered.groupby('order_id')['order_purchase_timestamp'].transform('first')
cutoff = order_purchase.quantile(0.8)
print('Test split cutoff:', cutoff)

full_train_items = delivered[order_purchase < cutoff].copy()
test_items = delivered[order_purchase >= cutoff].copy()

order_purchase = full_train_items.groupby('order_id')['order_purchase_timestamp'].transform('first')
cutoff = order_purchase.quantile(0.8)
print('Validation split cutoff:', cutoff)

validation_items = full_train_items[order_purchase >= cutoff].copy()
train_items = full_train_items[order_purchase < cutoff].copy()
print('train orders:', train_items['order_id'].nunique(), '| validation orders:', validation_items['order_id'].nunique(), '| test orders:', test_items['order_id'].nunique())

Test split cutoff: 2018-05-22 01:07:35
Validation split cutoff: 2018-03-17 11:50:35.200000
train orders: 60914 | validation orders: 15046 | test orders: 19110


**Splitting the data by purchase-date quantiles** (rather than using a randomized train-test split) **is the correct approach** for this use case. Delivery logistics, seller behavior, and shipping infrastructure naturally drift over time. Consequently, evaluating the model on randomly shuffled historical data would artificially inflate performance metrics, failing to reflect how the model will perform in production when processing future, unseen data.

## **1.2 creating lookup table based on the late delivery**

**A lookup table will be generated** to calculate historical seller lateness rates from prior order data. **This feature engineering step** incorporates additive smoothing (
𝐾𝑠𝑚𝑜𝑜𝑡ℎ=10) to stabilize predictions for low-volume merchants, preventing sellers with only one or two shipments from being skewed to extreme 0% or 100% rates. Furthermore, assigning the global lateness rate to a seller's first shipment serves as a robust, low-overhead validation that the time window is not peeking into the future.

In [5]:
K_SMOOTH = 10

full_train_items = full_train_items.sort_values(['seller_id', 'order_purchase_timestamp'])
full_train_items['seller_late_item'] = (
    full_train_items['shipping_limit_date'] < full_train_items['order_delivered_carrier_date']
).astype(int)

GLOBAL_LATE_RATE = full_train_items['seller_late_item'].mean()
print('global seller late-to-carrier rate (train):', round(GLOBAL_LATE_RATE, 4))

g = full_train_items.groupby('seller_id')['seller_late_item']
prior_count = g.cumcount()                                  
prior_late = g.cumsum() - full_train_items['seller_late_item'] 

full_train_items['seller_late_rate'] = (
    (prior_late + K_SMOOTH * GLOBAL_LATE_RATE) / (prior_count + K_SMOOTH)
)

first_shipment = full_train_items.groupby('seller_id').head(1)
assert np.allclose(first_shipment['seller_late_rate'], GLOBAL_LATE_RATE), 'expanding window leaked!'
print('expanding-window check passed: first shipment per seller == global rate')

global seller late-to-carrier rate (train): 0.0964
expanding-window check passed: first shipment per seller == global rate


In [6]:
seller_lookup = (
    full_train_items.groupby('seller_id')['seller_late_item']
    .agg(total_shipments='count', late_shipments='sum')
)
seller_lookup['seller_late_rate'] = (
    (seller_lookup['late_shipments'] + K_SMOOTH * GLOBAL_LATE_RATE)
    / (seller_lookup['total_shipments'] + K_SMOOTH)
)
print(seller_lookup.shape, 'sellers in lookup table')
seller_lookup.head()

(2401, 3) sellers in lookup table


,total_shipments,late_shipments,seller_late_rate
seller_id,,,
0015a82c2db000af6aaaf3ae2ecb0532,3,0,0.074123
001cca7ae9ae17fb1caed9dfb1094831,231,12,0.053791
002100f778ceb8431b7a1020ff7ab48f,54,5,0.093181
003554e2dce176b5555353e4f3555ac8,1,0,0.087599
004c9cd9d87a3c30c522c48c4fc07416,167,0,0.005444


**Sellers not present in the lookup table will be assigned the global lateness rate.** To validate this logic, the process will be evaluated against a sample dataset to quantify the volume of unseen sellers successfully imputed with the global baseline.

In [7]:
def attach_seller_late_rate_frozen(items_df, lookup, global_rate):
    merged = items_df.merge(
        lookup['seller_late_rate'], on='seller_id', how='left'
    )
    merged['seller_late_rate'] = merged['seller_late_rate'].fillna(global_rate)
    return merged

example = attach_seller_late_rate_frozen(test_items, seller_lookup, GLOBAL_LATE_RATE)
n_unseen = example['seller_id'].isin(seller_lookup.index).eq(False).sum()
print(f'test rows from sellers unseen in train: {n_unseen} (fell back to global rate)')

test rows from sellers unseen in train: 2964 (fell back to global rate)


## 1.3 to make the data as valid as possible

**Prior to constructing the model pipeline, the training data must be aligned** to match the exact schema and state of a real-time, incoming order. To achieve this environmental parity, we will subset the data to retain only the features available at the moment of order submission, transforming the dataset into the following structure:
| Column | Type | Description |
| --- | --- | --- |
| order_id | string | Unique identifier of the order (an order can have multiple items) |
| order_purchase_timestamp | datetime (string) | Timestamp when the order was placed. |
| product_id | string | Unique identifier of the product. |
| product_weight_g | float | Product weight in grams. |
| product_length_cm | float | Product length in cm. |
| product_height_cm | float | Product height in cm. |
| product_width_cm | float | Product width in cm. |
| seller_id | string | Unique identifier of the seller. |
| seller_zip_code_prefix | integer | First 5 digits of seller's zip code. |
| seller_geo_lat | float | Latitude of the seller zip code location. |
| seller_geo_lng | float | Longitude of the seller zip code location. |
| seller_city | string | Seller's city. |
| seller_state | string | Seller's state. |
| product_category_name_english | string | Same category translated to English. |
| customer_unique_id | string | The actual unique person (use this to track customers across multiple orders) |
| customer_zip_code_prefix | integer | First 5 digits of customer's zip code. |
| customer_geo_lat | float | Latitude of the customer zip code location. |
| customer_geo_lng | float | Longitude of the customer zip code location. |
| customer_city | string | Customer's city. |
| customer_state | string | Customer's state (Brazilian state code, e.g. SP, RJ). |
| order_estimated_delivery_date | datetime (string) | Delivery date estimate given to the customer at purchase time. |
| price | float | Item price. |
| freight_value | float | Freight/shipping cost for this item. (if an order has more than one item the freight value is splitted between items) |
| payment_sequential | integer | Sequence number if a customer used more than one payment method for the same order. |
| payment_type | string | Payment method (credit_card, boleto, voucher, debit_card). |
| n_vouchers | int | Total voucher used in that order |
| total_payment_value | float | Total payment paid for the order |
| payment_installments | integer | Number of installments chosen. |
| payment_value | float | Amount paid via this payment row. |

**Consequently, features that introduce data leakage will be dropped**, as they utilize information that would be unavailable at the time of inference in production.

In [8]:
DROP_COLUMNS =  ["customer_id", "order_status", "order_approved_at", "order_delivered_carrier_date", 
                 "order_delivered_customer_date", "order_item_id", "shipping_limit_date",
                 "review_score", "review_comment_message", "review_creation_date", "review_answer_timestamp"]

print('shape of train data before:', train_items.shape, ' | shape of test data before:', test_items.shape, ' | shape of validation data before:', validation_items.shape)

train_items.drop(columns=DROP_COLUMNS, inplace=True)
test_items.drop(columns=DROP_COLUMNS, inplace=True)
validation_items.drop(columns=DROP_COLUMNS, inplace=True)

print('shape of train data after:', train_items.shape, '  | shape of test data after:', test_items.shape, '  | shape of validation data after:', validation_items.shape)

shape of train data before: (69481, 40)  | shape of test data before: (21714, 40)  | shape of validation data before: (17371, 40)
shape of train data after: (69481, 29)   | shape of test data after: (21714, 29)   | shape of validation data after: (17371, 29)


## **1.4 Feature engineering**

**Prior to training the model, the dataset will be enriched with additional features** to improve predictive performance. The engineered features include the following:

### **1.4.1 Volume of the product**

**Product volume will be calculated as a composite feature** by combining the individual length, width, and height dimensions. This engineered feature consolidates dimensional attributes into a single, highly predictive signal for the model.

In [9]:
feat = delivered.copy()
feat['item_volume_cm3'] = (feat['product_length_cm'] * feat['product_height_cm'] * feat['product_width_cm'])

feat.loc[:, ["order_id", "item_volume_cm3"]].head()

,order_id,item_volume_cm3
0,e481f51cbdc54678b7cc49136f2d6af7,1976.0
1,53cdb2fc8bc7dce0b6741e2150273451,4693.0
2,47770eb9100c2d0c44946d9cf07ec65d,9576.0
3,949d5b44dbf5de918fe9c16f97b45f8a,6000.0
4,ad21c59c0840e6cb83a9ceb5573f8159,11475.0


### **1.4.2 Delivery Distance from Seller to Customer**

In [10]:
def calculate_haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1
    
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    r_earth = 6371
    return r_earth * c

**The Haversine formula will be utilized to calculate the geospatial distance** between sellers and customers, using the centroid latitude and longitude coordinates derived from their respective zip codes. This continuous distance feature will then be binned into categorical labels ranging from short-range to long-haul shipments.

In [11]:
feat["delivery_distance_km"] = calculate_haversine(feat["seller_lng"], feat["seller_lat"], feat["customer_lng"], feat["customer_lat"])

bins = [-1, 50, 300, 1000, 7000]
labels = ['Short-Range (<50km)', 'Regional (50-300km)', 'Interregional (300-1000km)', 'Long-Haul (>1000km)']

feat['distance_group'] = pd.cut(feat['delivery_distance_km'], bins=bins, labels=labels)

feat.loc[:, ["order_id", "distance_group"]].head()

,order_id,distance_group
0,e481f51cbdc54678b7cc49136f2d6af7,Short-Range (<50km)
1,53cdb2fc8bc7dce0b6741e2150273451,Interregional (300-1000km)
2,47770eb9100c2d0c44946d9cf07ec65d,Interregional (300-1000km)
3,949d5b44dbf5de918fe9c16f97b45f8a,Long-Haul (>1000km)
4,ad21c59c0840e6cb83a9ceb5573f8159,Short-Range (<50km)


### **1.4.3 Splitted Timestamp**

**Temporal features will be extracted from the order approval timestamp**, specifically isolating the day, month, and hour of approval. Finally, a promised_days column will be engineered to calculate the total duration allocated between the order creation and the promised delivery date.

In [12]:
feat['purchase_dow'] = feat['order_approved_at'].dt.day_name()
feat['purchase_month'] = feat['order_approved_at'].dt.month_name()
feat['purchase_hour'] = feat['order_approved_at'].dt.hour
feat['promised_days'] = (feat['order_estimated_delivery_date'] - feat['order_approved_at']).dt.days

feat.loc[:, ["order_id", 'purchase_dow', "purchase_month", 'purchase_hour', 'promised_days']].head()

,order_id,purchase_dow,purchase_month,purchase_hour,promised_days
0,e481f51cbdc54678b7cc49136f2d6af7,Monday,October,11,15
1,53cdb2fc8bc7dce0b6741e2150273451,Thursday,July,3,17
2,47770eb9100c2d0c44946d9cf07ec65d,Wednesday,August,8,26
3,949d5b44dbf5de918fe9c16f97b45f8a,Saturday,November,19,26
4,ad21c59c0840e6cb83a9ceb5573f8159,Tuesday,February,22,12


### **1.4.4 black friday or holiday or not**

**A binary holiday indicator will be created** to flag whether an order was placed during peak seasonal events, specifically targeting Black Friday and the December holiday shopping window.

In [13]:
def get_black_friday_date(year):
    november = pd.date_range(start=f'{year}-11-01', end=f'{year}-11-30', freq='D')
    fourth_thursday = november[november.weekday == 3][3]
    return (fourth_thursday + pd.Timedelta(days=1)).normalize()

def is_black_friday_or_holiday(ts, black_friday_window_days=2):
    black_friday = get_black_friday_date(ts.year)
    is_black_friday_period = abs((ts.normalize() - black_friday).days) <= black_friday_window_days
    is_december_holiday = ts.month == 12
    return bool(is_black_friday_period or is_december_holiday)

feat['is_black_friday_or_holiday'] = feat['order_purchase_timestamp'].apply(is_black_friday_or_holiday)

feat.loc[:, ["order_id", 'is_black_friday_or_holiday']].head()

,order_id,is_black_friday_or_holiday
0,e481f51cbdc54678b7cc49136f2d6af7,False
1,53cdb2fc8bc7dce0b6741e2150273451,False
2,47770eb9100c2d0c44946d9cf07ec65d,False
3,949d5b44dbf5de918fe9c16f97b45f8a,False
4,ad21c59c0840e6cb83a9ceb5573f8159,False


### **1.4.5 dominant order**

**Because the source data is structured at a multi-item level, new attributes will be engineered** to identify the primary seller, primary customer, and dominant product category for each unique order transaction. This step is critical to successfully aggregating the dataset from an item-level to a unified order-level format during downstream processing.

In [14]:
feat = attach_seller_late_rate_frozen(feat, seller_lookup, GLOBAL_LATE_RATE)
dominant = feat.loc[feat.groupby('order_id')['price'].idxmax()].set_index('order_id')

feat['dominant_product_category'] = feat['order_id'].map(dominant['product_category_name_english'])
feat['primary_seller_geo_state'] = feat['order_id'].map(dominant['seller_geo_state'])
feat['primary_seller_late_rate'] = feat['order_id'].map(dominant['seller_late_rate'])

feat.loc[:, ["order_id", 'dominant_product_category', "primary_seller_geo_state", 'primary_seller_late_rate']].head()

,order_id,dominant_product_category,primary_seller_geo_state,primary_seller_late_rate
0,e481f51cbdc54678b7cc49136f2d6af7,housewares,SP,0.015295
1,53cdb2fc8bc7dce0b6741e2150273451,perfumery,MG,0.026766
2,47770eb9100c2d0c44946d9cf07ec65d,auto,SP,0.040754
3,949d5b44dbf5de918fe9c16f97b45f8a,pet_shop,MG,0.070028
4,ad21c59c0840e6cb83a9ceb5573f8159,stationery,SP,0.038475


### **1.4.6 Information of order is interstate or intrastate**

**An interstate indicator and a state-to-state routing feature will be engineered** to capture regional logistics dynamics. The first feature will flag whether an order crosses state borders, while the second will explicitly map the geographical pipeline from the seller's state to the customer's destination state.

In [15]:
feat['is_interstate'] = feat['primary_seller_geo_state'] != feat['customer_geo_state']
feat['route'] = feat['primary_seller_geo_state'] + '_to_' + feat['customer_geo_state']

feat.loc[:, ["order_id", 'is_interstate', "route"]].head()

,order_id,is_interstate,route
0,e481f51cbdc54678b7cc49136f2d6af7,False,SP_to_SP
1,53cdb2fc8bc7dce0b6741e2150273451,True,MG_to_BA
2,47770eb9100c2d0c44946d9cf07ec65d,True,SP_to_GO
3,949d5b44dbf5de918fe9c16f97b45f8a,True,MG_to_RN
4,ad21c59c0840e6cb83a9ceb5573f8159,False,SP_to_SP


### **1.4.7 Aggregating Dataset**

**Following the item-level feature engineering steps, the dataset will be aggregated** to produce the final, order-level training set for the model. This aggregation process consolidates the multi-item records into a unified structure, ensuring that each row represents a unique order transaction ready for model ingestion.

In [16]:
out = pd.DataFrame({
    'total_price': feat.groupby('order_id')['price'].sum(),
    'total_freight_value': feat.groupby('order_id')['freight_value'].sum(),
    'total_weight_g': feat.groupby('order_id')['product_weight_g'].sum(),
    'total_volume_cm3': feat.groupby('order_id')['item_volume_cm3'].sum(),
    
    'n_items': feat.groupby('order_id').size(),
    
    'n_distinct_products': feat.groupby('order_id')['product_id'].nunique(),
    'n_distinct_categories': feat.groupby('order_id')['product_category_name_english'].nunique(),
    'n_distinct_sellers': feat.groupby('order_id')['seller_id'].nunique(),
    
    'distance_group': feat.groupby('order_id')['distance_group'].first(),
    
    'product_category': feat.groupby('order_id')['dominant_product_category'].first(),
    'seller_late_rate': feat.groupby('order_id')['primary_seller_late_rate'].first(),
    'order_purchase_timestamp': feat.groupby('order_id')['order_purchase_timestamp'].first(),
    'purchase_dow': feat.groupby('order_id')['purchase_dow'].first(),
    'purchase_month': feat.groupby('order_id')['purchase_month'].first(),
    'purchase_hour': feat.groupby('order_id')['purchase_hour'].first(),
    'promised_days': feat.groupby('order_id')['promised_days'].first(),
    'is_black_friday_or_holiday': feat.groupby('order_id')['is_black_friday_or_holiday'].first(),
    'order_estimated_delivery_date': feat.groupby('order_id')['order_estimated_delivery_date'].first(),
    'is_interstate': feat.groupby('order_id')['is_interstate'].first(),
    'route': feat.groupby('order_id')['route'].first(),
    'seller_geo_state': feat.groupby('order_id')['primary_seller_geo_state'].first(),
    'customer_geo_state': feat.groupby('order_id')['customer_geo_state'].first(),
    'payment_type': feat.groupby('order_id')['payment_type'].first(),
    'payment_installments': feat.groupby('order_id')['payment_installments'].first(),
    'total_payment_value': feat.groupby('order_id')['total_payment_value'].first(),
    'n_vouchers': feat.groupby('order_id')['n_vouchers'].first(),
    'voucher_value': feat.groupby('order_id')['voucher_value'].first(),
})

In [17]:
out.info()

<class 'pandas.DataFrame'>
Index: 95070 entries, 00010242fe8c5a6d1ba2dd792cb16214 to fffe41c64501cc87c801fd61db3f6244
Data columns (total 27 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   total_price                    95070 non-null  float64       
 1   total_freight_value            95070 non-null  float64       
 2   total_weight_g                 95070 non-null  float64       
 3   total_volume_cm3               95070 non-null  float64       
 4   n_items                        95070 non-null  int64         
 5   n_distinct_products            95070 non-null  int64         
 6   n_distinct_categories          95070 non-null  int64         
 7   n_distinct_sellers             95070 non-null  int64         
 8   distance_group                 95070 non-null  category      
 9   product_category               93722 non-null  str           
 10  seller_late_rate               95070 non

## **1.5 Pipeline**

### **1.5.1 Feature Fixing**

**Finally, all newly engineered features and aggregation steps will be encapsulated into a unified transformation function.** Designing the logic this way allows it to be seamlessly integrated into a production-ready data pipeline, streamlining downstream model deployment and inference.

In [23]:
def aggregate_order_features(items_df):
    feat = items_df.copy()
    feat = attach_seller_late_rate_frozen(feat, seller_lookup, GLOBAL_LATE_RATE)
    feat['item_volume_cm3'] = (
        feat['product_length_cm'] * feat['product_height_cm'] * feat['product_width_cm']
    )
    feat['seller_customer_distance_km'] = calculate_haversine(
        feat['seller_lat'], feat['seller_lng'], feat['customer_lat'], feat['customer_lng']
    )

    feat['purchase_dow'] = feat['order_purchase_timestamp'].dt.day_name()
    feat['purchase_month'] = feat['order_purchase_timestamp'].dt.month_name()
    feat['purchase_hour'] = feat['order_purchase_timestamp'].dt.hour
    feat['promised_days'] = (feat['order_estimated_delivery_date'] - feat['order_purchase_timestamp']).dt.days
    feat['is_black_friday_or_holiday'] = feat['order_purchase_timestamp'].apply(is_black_friday_or_holiday)
    
    dominant = feat.loc[feat.groupby('order_id')['price'].idxmax()].set_index('order_id')

    out = pd.DataFrame({
        'total_price': feat.groupby('order_id')['price'].sum(),
        'total_freight_value': feat.groupby('order_id')['freight_value'].sum(),
        'total_weight_g': feat.groupby('order_id')['product_weight_g'].sum(),
        'total_volume_cm3': feat.groupby('order_id')['item_volume_cm3'].sum(),
        
        'n_items': feat.groupby('order_id').size(),
        
        'n_distinct_products': feat.groupby('order_id')['product_id'].nunique(),
        'n_distinct_categories': feat.groupby('order_id')['product_category_name_english'].nunique(),
        'n_distinct_sellers': feat.groupby('order_id')['seller_id'].nunique(),
        
        'seller_customer_distance_km': feat.groupby('order_id')['seller_customer_distance_km'].max(),
        
        'order_purchase_timestamp': feat.groupby('order_id')['order_purchase_timestamp'].first(),
        'order_estimated_delivery_date': feat.groupby('order_id')['order_estimated_delivery_date'].first(),
        'purchase_dow': feat.groupby('order_id')['purchase_dow'].first(),
        'purchase_month': feat.groupby('order_id')['purchase_month'].first(),
        'purchase_hour': feat.groupby('order_id')['purchase_hour'].first(),
        'promised_days': feat.groupby('order_id')['promised_days'].first(),
        'is_black_friday_or_holiday': feat.groupby('order_id')['is_black_friday_or_holiday'].first(),
        'customer_geo_state': feat.groupby('order_id')['customer_geo_state'].first(),
        'payment_type': feat.groupby('order_id')['payment_type'].first(),
        'payment_installments': feat.groupby('order_id')['payment_installments'].first(),
        'total_payment_value': feat.groupby('order_id')['total_payment_value'].first(),
        'n_vouchers': feat.groupby('order_id')['n_vouchers'].first(),
        'voucher_value': feat.groupby('order_id')['voucher_value'].first(),
    })

    if 'is_late' in feat.columns:
        out['is_late'] = feat.groupby('order_id')['is_late'].first().astype(int)

    out['product_category'] = dominant['product_category_name_english'].reindex(out.index)
    out['seller_geo_state'] = dominant['seller_geo_state'].reindex(out.index)
    out['seller_late_rate'] = dominant['seller_late_rate'].reindex(out.index)

    bins = [-1, 50, 300, 1000, 7000]
    labels = ['Short-Range (<50km)', 'Regional (50-300km)', 'Interregional (300-1000km)', 'Long-Haul (>1000km)']
        
    out['distance_group'] = pd.cut(out['seller_customer_distance_km'], bins=bins, labels=labels)
    
    out['is_interstate'] = out['seller_geo_state'] != out['customer_geo_state']
    out['route'] = out['seller_geo_state'] + '_to_' + out['customer_geo_state']

    return out


In [24]:
class OrderFeatureAggregator(BaseEstimator, TransformerMixin):
    def __init__(self, seller_lookup=None, global_late_rate=None):
        self.seller_lookup = seller_lookup
        self.global_late_rate = global_late_rate

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return aggregate_order_features(X, self.seller_lookup, self.global_late_rate)

pipeline_engineer_step = FunctionTransformer(
    aggregate_order_features,
    kw_args={'lookup': seller_lookup, 'global_rate': GLOBAL_LATE_RATE},
)


### **1.5.2 Feature & Target**

**Although the data has been partitioned into training, validation, and testing sets, the features and target variables have not yet been isolated.** Consequently, each of the three datasets will be separated into distinct feature arrays (X) and target vectors (y) to prepare them for model training and evaluation.

In [25]:
def split_xy(df):
    agg = aggregate_order_features(df)
    y = agg.pop("is_late")
    X = agg
    return X, y

X_train, y_train = split_xy(train_items)
X_val, y_val = split_xy(validation_items)
X_test, y_test = split_xy(test_items)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "| y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "| y_test: ", y_test.shape)

X_train: (60914, 28) | y_train: (60914,)
X_val:   (15046, 28) | y_val:   (15046,)
X_test:  (19110, 28) | y_test:  (19110,)


### **1.5.3 Imputing & Encoding**

**Next, features will be categorized by data type to apply tailored imputation strategies.** Missing value imputation will be strictly confined to features influenced by seller data-entry habits, specifically `total_weight_g`, `total_volume_cm3`, and `product_category`. The remaining features will not be imputed, as they represent mandatory inputs populated automatically by the system; any missing values in those fields would indicate a fatal system error rather than typical data variance.

In [26]:
DROP_FOR_MODEL = [
    'order_purchase_timestamp', 'order_estimated_delivery_date', 'seller_customer_distance_km'
]

numeric_features = [
    'total_price', 'total_freight_value','n_items', 'n_distinct_products', 
    'n_distinct_categories', 'n_distinct_sellers',
    'seller_late_rate', 'purchase_hour', 
    'promised_days', 'payment_installments', 'total_payment_value',
    'n_vouchers', 'voucher_value',
]

oh_categorical_features = [
    'purchase_dow', 'purchase_month', 'payment_type', 'distance_group'
]

bin_categorical_features = [
    'route', 'seller_geo_state', 'customer_geo_state'
]

imputed_numeric_features = [
    'total_weight_g', 'total_volume_cm3'
]

imputed_categorical_features = ['product_category']

boolean_features = ['is_interstate', "is_black_friday_or_holiday"]

In [27]:
def convert_bool_to_int(df):
    return df.astype(int)

imputed_numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', RobustScaler()),
])

imputed_categorical_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encode', BinaryEncoder(handle_unknown='value')),
])

numeric_pipeline = Pipeline([
    ('scale', RobustScaler()),
])

oh_categorical_pipeline = Pipeline([
    ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False)),
])

bin_categorical_pipeline = Pipeline([
    ('encode', BinaryEncoder(handle_unknown='value')),
])

boolean_pipeline = Pipeline([
    ('transform', FunctionTransformer(convert_bool_to_int, validate=False, feature_names_out='one-to-one'))
])

preprocessor = ColumnTransformer([
    ('im_num', imputed_numeric_pipeline, imputed_numeric_features),
    ('im_cat', imputed_categorical_pipeline, imputed_categorical_features),
    ('num', numeric_pipeline, numeric_features),
    ('oh_cat', oh_categorical_pipeline, oh_categorical_features),
    ('bin_cat', bin_categorical_pipeline, bin_categorical_features),
    ('bool', boolean_pipeline, boolean_features),
], remainder='drop')

### **1.5.4 Full Pipeline**

In [28]:
pipeline = ImbPipeline([
    ('feature_fix', 'passthrough'),
    ('preprocessor', preprocessor),
    ('select', 'passthrough'),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])

pipeline

c:\Users\elmar\anaconda3\envs\Python_3.13\Lib\site-packages\sklearn\externals\_numpydoc\docscrape.py:420: UserWarning: Unknown section Example
  self[section] = content


,steps,"[('feature_fix', ...), ('preprocessor', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('im_num', ...), ('im_cat', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready fo

# **Section 2: Model Benchmarking**

In [29]:
def benchmark_models(pipeline, list_model, x_train, y_train, 
                     scoring="roc_auc", cv=5, random_state=None, select="passthrough"):

    all_cv_result = []

    tscv = TimeSeriesSplit(n_splits=cv)

    metric_names = [scoring] if isinstance(scoring, str) else list(scoring)

    for name, model in list_model.items():
        classifier = pipeline.set_params(classifier=model, select=select)
        cv_result = cross_validate(
            estimator=classifier,
            X=x_train, y=y_train,
            cv=tscv,
            scoring=scoring,
            return_train_score=True
        )

        row = {"name": name}
        for metric in metric_names:
            result_key = "score" if isinstance(scoring, str) else metric
            row[f"mean_train_{metric}"] = np.mean(cv_result[f"train_{result_key}"])
            row[f"std_train_{metric}"] = np.std(cv_result[f"train_{result_key}"])
            row[f"mean_test_{metric}"] = np.mean(cv_result[f"test_{result_key}"])
            row[f"std_test_{metric}"] = np.std(cv_result[f"test_{result_key}"])
        all_cv_result.append(row)

    sort_col = f"mean_test_{metric_names[0]}"
    result_df = pd.DataFrame(all_cv_result).sort_values(
        sort_col, ascending=False
    ).reset_index(drop=True)

    return result_df

## **2.1 Linear Based Model**

**Six linear-based algorithms will be benchmarked to select the optimal baseline model** for production deployment. This evaluation ensures the chosen model effectively addresses the underlying operational inefficiencies and solves the business problem at hand.

In [ ]:
list_linear_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Ridge Classifier": RidgeClassifier(random_state=RANDOM_STATE),
    "Linear SVM (LinearSVC)": LinearSVC(dual="auto", max_iter=5000, random_state=RANDOM_STATE),
    "Linear SVM (SVC Linear)": SVC(kernel="linear", random_state=RANDOM_STATE),
    "SGD Classifier": SGDClassifier(random_state=RANDOM_STATE),
    "Bagged Logistic Regression": BaggingClassifier(
        estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        random_state=RANDOM_STATE
    )
}

linear_select = SelectKBest(f_classif, k=20)

linear_benchmark_selected = benchmark_models(
    pipeline=pipeline,
    list_model=list_linear_models,
    x_train=X_train, y_train=y_train,
    cv=5, random_state=RANDOM_STATE,
    scoring="average_precision",
    select=linear_select,
)

display(linear_benchmark_selected)

## **2.2 Tree Based Model**

**In addition to the linear approaches, seven tree-based algorithms will be evaluated** to provide a robust comparative analysis. This dual-architecture benchmarking allows us to contrast linear and non-linear patterns, ensuring the highest-performing model is selected for final deployment.

In [ ]:
list_tree_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Extra Trees": ExtraTreesClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting (GBM)": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
}

tree_benchmark_selected = benchmark_models(
    pipeline=pipeline,
    list_model=list_tree_models,
    x_train=X_train, y_train=y_train,
    cv=5, random_state=RANDOM_STATE,
    scoring="average_precision"
)

display(tree_benchmark_selected)

# Section 3: Parameter Tuning

**Based on our baseline model benchmarking, which evaluated six linear and seven tree-based algorithms, Logistic Regression emerged as the top performer.** Consequently, we will perform hyperparameter tuning via Grid Search to further optimize the model's architecture and maximize predictive performance.

In [ ]:
grid_param_grid = [
    {
        'select': ['passthrough'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2', 'elasticnet'],
        'classifier__l1_ratio': [0.0, 0.5, 1.0],
        'classifier__max_iter': [2000],
    },
    {
        'select': [SelectKBest(f_classif)],
        'select__k': [15, 20, 30, 'all'],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2', 'elasticnet'],
        'classifier__l1_ratio': [0.0, 0.5, 1.0],
        'classifier__max_iter': [2000],
    },
]

tuning_pipeline = pipeline.set_params(
    classifier=LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', solver='saga'),
    smote='passthrough',
)

grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=grid_param_grid,
    scoring='average_precision',
    cv=TimeSeriesSplit(n_splits=5),
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train, y_train)

print("Best CV average_precision (grid search):", grid_search.best_score_)
print("Best params (grid search):", grid_search.best_params_)


In [ ]:
best_pipeline = grid_search.best_estimator_

val_proba_uncalibrated = best_pipeline.predict_proba(X_val)[:, 1]
print("Validation average_precision (uncalibrated):",
      average_precision_score(y_val, val_proba_uncalibrated))

# Section 4: Learning Curve

In [ ]:
calibrated_pipeline = CalibratedClassifierCV(
    estimator=FrozenEstimator(best_pipeline),
    method='isotonic'
)
calibrated_pipeline.fit(X_val, y_val)

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    estimator=best_pipeline,
    X=X_train, y=y_train,
    cv=TimeSeriesSplit(n_splits=5),
    scoring='average_precision',
    train_sizes=np.linspace(0.1, 1.0, 10),
    shuffle=False,           
    n_jobs=-1,
    random_state=RANDOM_STATE,
    error_score=np.nan
)

train_mean, train_std = np.nanmean(train_scores, axis=1), np.nanstd(train_scores, axis=1)
val_mean, val_std = np.nanmean(val_scores, axis=1), np.nanstd(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', color='#1f77b4', label='Training score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#1f77b4')

plt.plot(train_sizes, val_mean, 'o-', color='#d62728', label='Validation score (CV)')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#d62728')

plt.xlabel('Training examples')
plt.ylabel('Average precision')
plt.title('Learning Curve: is the model data-limited?')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**The model is not data-limited, but it suffers from severe underfitting and an anomalous performance gap.** Expanding the training dataset beyond 5,000 examples yields diminishing returns. 

* **Performance Plateau:** Both metrics flatten completely after **5,000 samples**, proving additional data will not improve the current configuration.
* **Severe Underfitting:** The average precision remains critically low (**< 0.18**), showing the model fails to capture the underlying problem complexity.
* **Inverse Performance Gap:** The validation score is consistently *higher* than the training score, which is highly atypical.

# Section 5: Threshold Optimization

**Aligned with the objectives defined in the Business Understanding section, our primary goal is to maximize recall while optimizing precision** to mitigate the operational cost of excessive false alarms. To achieve this balance, we will perform threshold optimization to identify the ideal decision boundary that satisfies both requirements.

In [ ]:
y_val_proba = calibrated_pipeline.predict_proba(X_val)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)

MIN_PRECISION = 0.1

precisions_t, recalls_t = precisions[:-1], recalls[:-1]

meets_floor = precisions_t >= MIN_PRECISION

if meets_floor.any():
    candidate_recalls = np.where(meets_floor, recalls_t, -1)
    best_idx = np.argmax(candidate_recalls)
    best_threshold = thresholds[best_idx]
    best_precision = precisions_t[best_idx]
    best_recall = recalls_t[best_idx]
    best_f1 = 2 * best_precision * best_recall / (best_precision + best_recall + 1e-9)

    print(f"Best threshold (max recall subject to precision >= {MIN_PRECISION:.2f}): {best_threshold:.3f}")
    print(f"  Precision: {best_precision:.3f}")
    print(f"  Recall:    {best_recall:.3f}")
    print(f"  F1:        {best_f1:.3f}")
else:
    print(f"No threshold reaches precision >= {MIN_PRECISION:.2f}.")
    print(f"Max achievable precision at any threshold is {precisions_t.max():.3f} -- lower MIN_PRECISION and rerun.")

plt.figure(figsize=(7, 5))
plt.plot(thresholds, precisions_t, label='Precision')
plt.plot(thresholds, recalls_t, label='Recall')
plt.axhline(MIN_PRECISION, color='gray', linestyle='--', linewidth=1, label='MIN_PRECISION floor')
if meets_floor.any():
    plt.axvline(best_threshold, color='red', linestyle=':', linewidth=1, label='Chosen threshold')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision & Recall vs. Threshold (validation set)')
plt.legend()
plt.show()

**Upon determining the optimal decision threshold, we will construct confusion matrices to compare performance against the default baseline.** This comparative analysis will visually demonstrate how threshold tuning shifts the balance of true positives and false alarms, validating its alignment with our business objectives.

In [ ]:
y_val_proba = calibrated_pipeline.predict_proba(X_val)[:, 1]
y_test_proba = calibrated_pipeline.predict_proba(X_test)[:, 1]

y_val_pred_untuned = (y_val_proba >= 0.5).astype(int)
y_test_pred_untuned = (y_test_proba >= 0.5).astype(int)

y_val_pred_tuned = (y_val_proba >= best_threshold).astype(int)
y_test_pred_tuned = (y_test_proba >= best_threshold).astype(int)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes_flat = axes.ravel()

cmu_val = confusion_matrix(y_val, y_val_pred_untuned)
cmu_test = confusion_matrix(y_test, y_test_pred_untuned)
cm_val = confusion_matrix(y_val, y_val_pred_tuned)
cm_test = confusion_matrix(y_test, y_test_pred_tuned)

ConfusionMatrixDisplay(cmu_val, display_labels=["not late", "is late"]).plot(
    ax=axes_flat[0], colorbar=False, cmap="Blues"
)
axes_flat[0].set_title("Confusion Matrix - Validation")

ConfusionMatrixDisplay(cmu_test, display_labels=["not late", "late"]).plot(
    ax=axes_flat[1], colorbar=False, cmap="Blues"
)
axes_flat[1].set_title("Confusion Matrix - Testing")

ConfusionMatrixDisplay(cm_val, display_labels=["not late", "is late"]).plot(
    ax=axes_flat[2], colorbar=False, cmap="Blues"
)
axes_flat[2].set_title("Tuned Confusion Matrix - Validation")

ConfusionMatrixDisplay(cm_test, display_labels=["not late", "late"]).plot(
    ax=axes_flat[3], colorbar=False, cmap="Blues"
)
axes_flat[3].set_title("Tuned Confusion Matrix - Testing")

plt.tight_layout()
plt.show()


**The confusion matrix results indicate that the tuned threshold significantly enhances the model's predictive capacity.** Although this optimization led to a noticeable increase in false alarms, it directly aligns with our core objective to capture high-risk late orders. Specifically, true positive identifications escalated from **32 to 1,262** within the validation dataset, and from **1 to 931** within the test dataset, demonstrating that the operational trade-off successfully prioritizes risk mitigation.

# Section 6: Model Explainability

In [ ]:

preprocessor_fitted = best_pipeline.named_steps['preprocessor']
select_fitted = best_pipeline.named_steps['select']
classifier = best_pipeline.named_steps['classifier']

X_train_preprocessed = preprocessor_fitted.transform(X_train)
X_test_preprocessed = preprocessor_fitted.transform(X_test)

try:
    all_feature_names = preprocessor_fitted.get_feature_names_out()
except Exception:
    all_feature_names = np.array([f"feature_{i}" for i in range(X_train_preprocessed.shape[1])])

all_feature_names = np.array([name.split('__', 1)[-1] for name in all_feature_names])

if select_fitted == 'passthrough':
    X_train_transformed = X_train_preprocessed
    X_test_transformed = X_test_preprocessed
    feature_names = all_feature_names
else:
    X_train_transformed = select_fitted.transform(X_train_preprocessed)
    X_test_transformed = select_fitted.transform(X_test_preprocessed)
    feature_names = all_feature_names[select_fitted.get_support()]

feature_names = np.asarray(feature_names, dtype=str)

X_train_transformed = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names)

print("Feature selection step in best_pipeline:", select_fitted)
print("Transformed feature matrix shape (post feature-selection):", X_test_transformed.shape)


## 6.1 Log-Odds Coefficients & Odds Ratios

In [ ]:
coefs = classifier.coef_.ravel()
odds_ratios = np.exp(coefs)

coef_table = pd.DataFrame({
    'feature': feature_names,
    'log_odds_coef': coefs,
    'odds_ratio': odds_ratios,
}).sort_values('log_odds_coef', key=np.abs, ascending=False).reset_index(drop=True)

coef_table


In [ ]:
top_n = min(20, len(coef_table))
plot_df = coef_table.head(top_n).sort_values('log_odds_coef')

plt.figure(figsize=(8, max(4, top_n * 0.3)))
colors = ['#d62728' if c > 0 else '#1f77b4' for c in plot_df['log_odds_coef']]
plt.barh(plot_df['feature'], plot_df['log_odds_coef'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Log-odds coefficient')
plt.title(f'Top {top_n} features by |log-odds coefficient| (red = pushes toward late)')
plt.tight_layout()
plt.show()


## 6.2 Global feature importance (SHAP)

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)

background_idx = rng.choice(X_train_transformed.shape[0], size=min(200, X_train_transformed.shape[0]), replace=False)
background = X_train_transformed.iloc[background_idx]

explainer = shap.LinearExplainer(classifier, background)

sample_idx = rng.choice(X_test_transformed.shape[0], size=min(2000, X_test_transformed.shape[0]), replace=False)
X_explain = X_test_transformed.iloc[sample_idx]

shap_values_array = explainer.shap_values(X_explain)
expected_value = explainer.expected_value
base_values = np.full(len(X_explain), expected_value) if np.ndim(expected_value) == 0 else expected_value

shap_values = shap.Explanation(
    values=shap_values_array,
    base_values=base_values,
    data=X_explain.values,
    feature_names=list(feature_names),
)


In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)

In [ ]:
shap.plots.bar(shap_values, max_display=20)

## 6.3 Local explanation: a correctly-flagged late order

In [ ]:
test_proba_sample = classifier.predict_proba(X_explain)[:, 1]

y_test_sample = y_test.reset_index(drop=True).iloc[sample_idx].reset_index(drop=True)

true_positive_mask = (y_test_sample.values == 1) & (test_proba_sample >= best_threshold)

if true_positive_mask.any():
    local_pos = np.argmax(np.where(true_positive_mask, test_proba_sample, -1))
else:
    print("No true positive in this sample at the chosen threshold -- showing the highest-risk order instead.")
    local_pos = int(np.argmax(test_proba_sample))

print(f"Model probability for this order: {test_proba_sample[local_pos]:.3f} "
      f"(threshold: {best_threshold:.3f}) | Actual label: {'late' if y_test_sample.iloc[local_pos] == 1 else 'not late'}")

shap.plots.waterfall(shap_values[local_pos], max_display=15)


# **Section 7: Deployment**

## 7.1 Refreshable seller lookup table

In [ ]:
def build_seller_lookup(items_df, k_smooth=10):
    items_df = items_df.sort_values(['seller_id', 'order_purchase_timestamp']).copy()
    items_df['seller_late_item'] = (
        items_df['shipping_limit_date'] < items_df['order_delivered_carrier_date']
    ).astype(int)

    global_late_rate = items_df['seller_late_item'].mean()

    seller_lookup = (
        items_df.groupby('seller_id')['seller_late_item']
        .agg(total_shipments='count', late_shipments='sum')
    )
    seller_lookup['seller_late_rate'] = (
        (seller_lookup['late_shipments'] + k_smooth * global_late_rate)
        / (seller_lookup['total_shipments'] + k_smooth)
    )
    return seller_lookup, global_late_rate


In [ ]:
SELLER_LOOKUP_DIR = '../Dataset/Processed Data'
os.makedirs(SELLER_LOOKUP_DIR, exist_ok=True)

def save_seller_lookup(seller_lookup, global_late_rate, out_dir=SELLER_LOOKUP_DIR):
    tag = date.today().isoformat()
    dated_path = f'{out_dir}/seller_lookup_{tag}.joblib'
    latest_path = f'{out_dir}/seller_lookup_latest.joblib'

    bundle = {'seller_lookup': seller_lookup, 'global_late_rate': global_late_rate}
    joblib.dump(bundle, dated_path)
    joblib.dump(bundle, latest_path)
    return dated_path

def load_seller_lookup(path=f'{SELLER_LOOKUP_DIR}/seller_lookup_latest.joblib'):
    bundle = joblib.load(path)
    return bundle['seller_lookup'], bundle['global_late_rate']

def refresh_seller_lookup(latest_items_df, k_smooth=10):
    lookup, global_rate = build_seller_lookup(latest_items_df, k_smooth)
    path = save_seller_lookup(lookup, global_rate)
    print(f'Refreshed seller lookup: {lookup.shape[0]} sellers, saved to {path}')
    return lookup, global_rate


## 7.2 Saving Model (without Lookup inserted)

In [ ]:
MODEL_LOOKUP_DIR = '../Assets/Model'
os.makedirs(MODEL_LOOKUP_DIR, exist_ok=True)

deploy_pipeline_for_export = copy.deepcopy(calibrated_pipeline)
deploy_pipeline_for_export.set_params(feature_fix='passthrough')

with open(f'{MODEL_LOOKUP_DIR}/deploy_pipeline.pkl', 'wb') as f:
    pickle.dump(deploy_pipeline_for_export, f)

save_seller_lookup(seller_lookup, GLOBAL_LATE_RATE)
print(f'Saved {MODEL_LOOKUP_DIR}/deploy_pipeline.pkl and initial seller lookup snapshot')


## 7.3 Serving: combine the two artifacts

In [ ]:
def load_serving_pipeline(model_path=f'{MODEL_LOOKUP_DIR}/deploy_pipeline.pkl',
                           lookup_path=f'{SELLER_LOOKUP_DIR}/seller_lookup_latest.joblib'):
    pipeline_ = pickle.load(open(model_path, 'rb'))
    lookup, global_rate = load_seller_lookup(lookup_path)

    pipeline_.set_params(
        feature_fix=OrderFeatureAggregator(
            seller_lookup=lookup,
            global_late_rate=global_rate,
        )
    )
    return pipeline_

serving_pipeline = load_serving_pipeline()
serving_pipeline


### Sanity check: raw, item-level rows in -> probability out

In [ ]:
raw_sample = test_items.head(50)

sanity_proba = serving_pipeline.predict_proba(raw_sample)[:, 1]
print('Serving pipeline output shape:', sanity_proba.shape)
print(sanity_proba[:10])
